<a name='overview'></a>
# Overview

This notebook presents a systematic benchmark comparing two instance segmentation systems — **YOLO11-seg (Ultralytics)** and **SAM 3 (Segment Anything Model 3, Meta)** — across different hardware configurations and model sizes.

Rather than evaluating accuracy in isolation, the goal is to characterize the full latency, throughput, memory, and detection behavior of each system under realistic video inference conditions.

The benchmark was run on a 200-frame video sequence and repeated across CPU and GPU backends.

Three YOLO11-seg model sizes were tested (yolo11n-seg, yolo11m-seg, yolo11x-seg) alongside a single SAM 3 configuration running on GPU, **prompted with the concepts `["person", "car"]`** — SAM 3 is *concept-conditioned*, not a class-agnostic "segment everything" model.

> **Generations.** YOLO11 (Sep 2024) is Ultralytics' mature default and is what we benchmark. **YOLO26** (Jan 2026) is the current generation: natively end-to-end / NMS-free, DFL removed, and up to ~43% faster on CPU ONNX than YOLO11n. Several conclusions below are about the NMS + mask-postprocessing stage that YOLO26 removes — exercise 4 asks you to re-measure without it. SAM 3 (Nov 2025) is the current Segment Anything generation.

> ⚠️ **Every measured number quoted in the prose comes from one reference run** (Colab T4, 200 frames of `shinjuku.mp4`). Tables and figures are regenerated from `all_results` when you re-run the notebook; the narrative text is not. Where the two disagree, the generated table is right.

## Contents

- [Overview](#overview)
- [Setup & Installation](#setup)
  - [Benchmark protocol](#protocol)
1. [Tracking algorithms — ByteTrack vs BoT-SORT](#tracking)
2. [YOLO11 — Instance Segmentation + Multi-Object Tracking](#yolo11)
3. [SAM 3](#sam3)
4. [YOLO11 (CPU & GPU) vs SAM 3 — Summary](#summary)
5. [Conclusions](#conclusions)
6. [Think & Exercise](#exercises)

<a name='setup'></a>
# Setup & Installation

In [ ]:
# Install dependencies (run once).
# Colab already ships torch, torchvision, numpy, opencv, matplotlib, pandas, seaborn and gdown
# with a CUDA-matched build. Reinstalling torch/torchvision from PyPI breaks the GPU runtime,
# so we only add what Colab is missing.
!pip install -q ultralytics psutil gputil

## Download video

In [ ]:

import gdown

file_id = "13d-WecRvyTozDcQrHxLV1t_e5GQzPiPz"
video_path = '/content/shinjuku.mp4'

url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, video_path, quiet=False)


## Preprocessed Example mp4 file

ex_file_id = "1Yy74XqnVOCdRxBxSuw7o0X4iI0BL2Wav"
video_path_preprocessed = '/content/shinjuku_preprocessed_ex.mp4'

url = f'https://drive.google.com/uc?id={ex_file_id}'
gdown.download(url, video_path_preprocessed, quiet=False)

## Imports

In [ ]:
import os
import time
import json
import warnings
import numpy as np
import cv2
import torch
import psutil
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

warnings.filterwarnings('ignore')

# ── Output dirs ──────────────────────────────────────────────
Path('results').mkdir(exist_ok=True)
Path('output_videos').mkdir(exist_ok=True)

# ── Device availability ───────────────────────────────────────
HAS_CUDA = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else 'N/A'

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {HAS_CUDA}')
print(f'GPU      : {GPU_NAME}')
print(f'CPU cores: {psutil.cpu_count(logical=True)}')

## Configs

In [ ]:
# ─── USER CONFIG ─────────────────────────────────────────────────────────────
VIDEO_PATH     = video_path
MAX_FRAMES     = 200                # set None to process full video
CONF_THRESH    = 0.30
IOU_THRESH     = 0.45

# Ultralytics ships SIX trackers (since 8.4.63):
#   bytetrack.yaml   botsort.yaml   ocsort.yaml
#   deepocsort.yaml  fasttrack.yaml tracktrack.yaml   (TrackTrack, CVPR 2025)
# The shipped default has moved: cfg/default.yaml on main reads `tracker: tracktrack.yaml`
# while the docs still say botsort.yaml — so never assume it, check the version you installed:
#   from ultralytics.cfg import get_cfg; print(get_cfg().tracker)
TRACKER        = 'bytetrack.yaml'

CLASS_FILTER   = None               # e.g. [0] for 'person' only, None = all
SAVE_VIDEO     = True
# ─────────────────────────────────────────────────────────────────────────────

DEVICES = {'CPU': 'cpu'}
if HAS_CUDA:
    DEVICES['GPU'] = 0
print('Devices to benchmark:', list(DEVICES.keys()))

import ultralytics
print('ultralytics:', ultralytics.__version__)

## Load Video

We are going to use the video /content/shinjuku.mp4, which has a resolution of 1280×606. The video contains 631 frames and runs at 30.0 FPS.

**Q: If the video has 631 frames at 30 FPS, what is its total duration?**

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS_SRC      = cap.get(cv2.CAP_PROP_FPS)
W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Video: {VIDEO_PATH}  |  {W}x{H}  |  {TOTAL_FRAMES} frames  |  {FPS_SRC:.1f} fps')

## GPU memory helpers

In [ ]:
def get_gpu_memory_mb() -> float:
    """Allocated VRAM in MB (0 if no CUDA)."""
    if HAS_CUDA:
        return torch.cuda.memory_allocated() / 1e6
    return 0.0

def get_cpu_ram_mb() -> float:
    """Current process RSS in MB."""
    return psutil.Process().memory_info().rss / 1e6

def reset_gpu_stats():
    if HAS_CUDA:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

<a name='protocol'></a>
## Benchmark protocol

A benchmark is only a benchmark if both sides are measured the same way. Before any number in this
notebook means anything, we fix one protocol and apply it to **both** YOLO11 and SAM 3.

| Knob | Value here | Why it matters |
|---|---|---|
| Input resolution | `IMGSZ = 640`, passed explicitly to **both** models | Latency scales roughly quadratically with `imgsz`. An unrecorded resolution makes the comparison meaningless. SAM 3's native size is 1008 (patch 14 → 72×72 grid); Ultralytics' own `SAM3VideoSemanticPredictor` example runs at 640, so 640 is a legitimate shared setting — exercise 6 asks what 1008 costs. |
| Precision | `QUANTIZE = 16` (FP16) on GPU for both; FP32 on CPU, which has no FP16 kernel path — recorded per run | FP16 vs FP32 is worth roughly 2× on a tensor-core GPU. Timing FP16 against the FP32 default is not a model comparison. |
| Temporal association | on for both (YOLO11 + ByteTrack; SAM 3's video predictor tracks with its memory bank) | Tracking is part of the per-frame cost. Either both pay it or neither does. |
| Warm-up | `WARMUP_FRAMES = 10`, discarded | The first frames pay CUDA context creation, cuDNN autotuning and lazy kernel compilation. |
| Frames measured | `MAX_FRAMES = 200` after warm-up | Same frames, same order, for every configuration. |
| GPU timing | `torch.cuda.synchronize()` immediately before **every** timestamp | CUDA kernels are asynchronous. Without a sync you time the kernel *launch*, not the work. |
| CPU threads | `torch.get_num_threads()` printed and stored in every summary | CPU latency is roughly inversely proportional to thread count; a CPU number without it is unreproducible. |
| Scope | `mean_fwd_ms` = forward pass only · `mean_ms` = wall clock around the whole call · `mean_total_ms` = preprocess + forward + postprocess | These are **different quantities**. Only compare like with like. |

### The scope trap

The classic mistake in exactly this comparison is to wall-clock one model (preprocess + forward +
NMS + mask upsampling + tracker association) while reading the other model's
`results.speed["inference"]` (forward pass only), and then to publish the ratio as a speedup. Both
quantities are recorded below for both models, and every comparison in this notebook says which one
it is using.

A second, quieter trap: `perf_counter()` around a CUDA call measures how long it took to *queue*
the kernels unless you synchronize. On a fast GPU that can under-report by an order of magnitude,
which is exactly the regime where the mistake is hardest to notice.

In [ ]:
# ─── SHARED BENCHMARK PROTOCOL ───────────────────────────────────────────────
IMGSZ          = 640     # same input resolution for every timed run
QUANTIZE       = 16      # 16 = FP16 (GPU only), 32 = FP32. Same precision for every timed run.
WARMUP_FRAMES  = 10      # discarded before timing starts
TRACKING       = True    # temporal association enabled on both sides
# ─────────────────────────────────────────────────────────────────────────────


def sync():
    """Block until every queued CUDA kernel has finished, so perf_counter() times real work."""
    if HAS_CUDA:
        torch.cuda.synchronize()


N_THREADS = torch.get_num_threads()
print(f'CPU threads (intra-op): {N_THREADS}')
print(f'CPU threads (inter-op): {torch.get_num_interop_threads()}')
print(f'imgsz={IMGSZ}  quantize=FP{QUANTIZE} (GPU) / FP32 (CPU)  '
      f'warmup={WARMUP_FRAMES}  frames={MAX_FRAMES}  tracking={TRACKING}')

<a name='tracking'></a>
# 1 - Tracking algorithms (ByteTrack vs BoT-SORT)

Ultralytics ships **six** trackers today — `bytetrack`, `botsort`, `ocsort`, `deepocsort`,
`fasttrack` and `tracktrack` (the last four added in 8.4.63; TrackTrack is CVPR 2025). We study the
two classical ones in depth because every other tracker in the list is a variation on the same
loop, and we benchmark with ByteTrack. Section 1.5 lists what the other four change.

## How both work (shared foundations)

Both algorithms follow the same general loop every frame:

1. Run a detector (YOLO, etc.) to get bounding boxes + confidence scores
2. Use a **[Kalman filter](https://en.wikipedia.org/wiki/Kalman_filter)** to predict where existing tracks will be this frame
3. **Associate** detections to tracks using a cost matrix + [Hungarian algorithm](https://en.wikipedia.org/wiki/Hungarian_algorithm).

4. Update matched tracks, initialize new ones, delete lost ones

The key innovations are *how* they handle the association step.



## 1.1 Hungarian algorithm

The **Hungarian algorithm** is a method used in **optimization problems**—specifically, it solves the **assignment problem**.

### What is the assignment problem?

It’s about assigning a set of tasks to a set of agents (like workers, machines, or jobs) in a way that:

* Each agent gets exactly one task
* Each task is assigned to exactly one agent
* The **total cost is minimized** (or total profit is maximized)

---

### Core idea

The Hungarian algorithm finds the **optimal assignment** by working with a **cost matrix** (a table where each entry represents the cost of assigning a specific agent to a specific task).

---

### How it works (simplified steps)

1. **Create the cost matrix**
   Rows = agents, columns = tasks.

2. **Row reduction**
   Subtract the smallest value in each row from all elements of that row.

3. **Column reduction**
   Subtract the smallest value in each column from all elements of that column.

4. **Cover all zeros with minimum number of lines**
   Use horizontal/vertical lines to cover all zero values.

5. **Check optimality**

   * If number of lines = number of rows (or columns), you have an optimal assignment.
   * If not, adjust the matrix and repeat.

6. **Make assignments**
   Choose zeros such that no two are in the same row or column.

---

### Why it’s useful

* Guarantees the **optimal solution**
* Runs in polynomial time (efficient compared to brute force)
* Widely used in:

  * Job scheduling
  * Resource allocation
  * Matching problems (e.g., workers ↔ tasks)

---
Example (simple idea)
---

Each cell shows how many minutes driver takes to reach that car.

**Goal**: find the assignment that minimizes total time.


<u>The problem</u>
---
Assign each driver to exactly one car, minimizing total travel time . **Hungarian is O(n³)**.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-0.png">

Step 1 — row reduction
---
Subtract the minimum value in each row from all entries in that row.

**This guarantees at least one zero per row.**

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-1.png">

The total cost doesn't change — subtracting a constant from a row shifts all assignments equally.

Step 2 — column reduction
---

Subtract the minimum value in each column from all entries in that column.

**Adds more zeros for the algorithm to work with.**

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-2.png">

Now we have 6 zeros in the matrix.

Step 3 — cover all zeros with minimum lines
---
Find the fewest lines (horizontal or vertical) that cover every zero.

If the number of lines equals n=4, we can read the optimal assignment.

Here we need only 3 lines — **not enough yet**.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-4.png">

Since 3 < 4, the current matrix doesn't yet have enough independent zeros for a full assignment.

Min uncovered value = 1

Step 4 — adjust the matrix
---

Subtract the minimum uncovered value (1) from all **uncovered cells**.

Add it to **doubly-covered** cells.

Singly-covered cells stay the same.
This creates new zeros without breaking existing ones.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-5.png">

Step 5 — optimal assignment found!
---

We can now cover all zeros with 4 lines (= n).

Find a perfect matching: one zero per row and per column.

Total time = 3 + 8 + 5 + 9 = 25 minutes.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/hungarian-6.png">


Alex → Car A  3 min
Ben → Car C  8 min
Chloe → Car B  5 min
Daisy → Car D  9 min
Total: 25 min ✓

Best possible — any other combo takes longer.


## 1.2 The Kalman filter

**The core idea in one sentence:** a Kalman filter is just a *weighted average* of two noisy estimates — your model's prediction and the sensor's reading — where the weights are determined by how uncertain each source is.

**The predict step** uses your motion model (physics, constant velocity, whatever you know) to project the current belief forward in time. Uncertainty always *grows* here because models are imperfect (process noise Q).

This is why a Kalman filter never "freezes" — it keeps accounting for the fact that the world keeps moving.

**The update step**
---
It is where the magic happens.

When a measurement z arrives, you compute the Kalman gain:

```
K = σ²_pred / (σ²_pred + R)
```

K is a number between 0 and 1.

If your prediction is very uncertain (large σ²) and the sensor is precise (small R), K → 1 and you move almost entirely to the measurement.

If the opposite, K → 0 and you mostly ignore the sensor.
The new mean and variance:

```
μ_post  = μ_pred + K · (z − μ_pred)     ← weighted correction
σ²_post = (1 − K) · σ²_pred             ← always smaller than σ²_pred
```

The posterior variance `(1-K)·σ²_pred` is *always smaller* than either input variance.

This is the algebraic proof that fusing two uncertain sources makes you more certain.

---
**Why it works for tracking (ByteTrack / BoT-SORT connection)**
---

In multi-object tracking the state is **8-dimensional**, not 4: it carries the box *shape* as well
as its position, plus the velocity of every component.

| Tracker | Filter | State |
|---|---|---|
| ByteTrack | `KalmanFilterXYAH` | `[x, y, a, h, vx, vy, va, vh]` — centre, aspect ratio `a = w/h`, height |
| BoT-SORT | `KalmanFilterXYWH` | `[x, y, w, h, vx, vy, vw, vh]` — centre, width, height |

This matters for the next section: a `[x, y, vx, vy]` state would predict a *point*, and you cannot
compute an IoU cost against a point. Because size is part of the state, `predict` returns a full
box that drops straight into the IoU cost matrix — and box dimensions get smoothed across frames
as a bonus.

The `predict` step uses the constant-velocity model to guess where the bounding box will be next frame.

The `update` step fuses that with the detector's actual bounding box output.

This is what lets trackers survive frames where the detector misses an object — the Kalman prediction keeps the track alive.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_7_CPU_vs_GPU_YOLOV11_vs_SAM3/kalman-filter.png">

---

## 1.3 ByteTrack — every detection counts

ByteTrack's core idea: **don't throw away low-confidence detections**. Standard trackers discard anything below a score threshold, losing useful signal for occluded or partially visible objects.

Instead, ByteTrack runs **two association passes**:

**Pass 1** — match tracks to *high-confidence* detections (score ≥ τ_high, typically 0.6) using IoU-based cost + Hungarian algorithm. Most tracks get resolved here.

**Pass 2** — take unmatched tracks from Pass 1, and try to match them against *low-confidence* detections (τ_low ≤ score < τ_high). This is where occluded objects hiding behind partially visible detections get recovered.

Identity is maintained **purely through motion** (Kalman + IoU). No appearance features, no camera model — which is why it's extremely fast.

---

## 1.4 BoT-SORT — motion + camera compensation (+ optional appearance)

In Ultralytics, BoT-SORT is declared as `class BOTSORT(BYTETracker)`: it **inherits** ByteTrack's
two-pass high/low-score association and adds three things on top. It is not an alternative to
ByteTrack's idea — it is ByteTrack plus extras.

**1. Camera motion breaks IoU.** If the camera pans, every track's Kalman prediction is wrong
relative to the frame — IoU collapses even for correctly tracked objects. BoT-SORT adds **Global
Motion Compensation (GMC)**: it estimates the frame-to-frame camera warp and applies it to the
Kalman state before association. Ultralytics' `botsort.yaml` ships **`gmc_method: sparseOptFlow`**
(sparse optical flow, as in the paper). `ecc` (Enhanced Correlation Coefficient), `orb` and `none`
are selectable alternatives — ECC is the most accurate of them and by far the slowest, which is why
it is not the default.

**2. A better Kalman state.** BoT-SORT swaps ByteTrack's `KalmanFilterXYAH` (`x, y, a, h`) for
`KalmanFilterXYWH` (`x, y, w, h`), estimating width and height directly instead of aspect ratio and
height. This removes a whole class of box-shape drift, and it is the contribution most often left
out of comparison tables.

**3. Appearance (ReID) — optional, and off by default.** BoT-SORT *can* fuse an appearance
embedding into the cost matrix:

```
cost = α · (1 - IoU) + (1 - α) · cosine_distance(ReID_query, ReID_gallery)
```

> ⚠️ `botsort.yaml` ships **`with_reid: False`**. Out of the box, the BoT-SORT you get from
> Ultralytics is motion + GMC only, and the formula above does **not** describe what runs on your
> video. Set `with_reid: True` (and choose a `model:`, e.g. `auto` or a `.pt` feature extractor) to
> enable the appearance term — and expect to pay for it in latency, because you are now running a
> second network per detection.

---

## 1.5 Comparison table

These rows describe the configs **Ultralytics actually ships** (`bytetrack.yaml`,
`botsort.yaml`) — not the papers' best-case settings. If you pick a tracker from a table, pick it
from a table that matches your config file.

| Property | ByteTrack | BoT-SORT (shipped config) |
|---|---|---|
| Low-score recovery | ✅ Two-pass split | ✅ Same — `BOTSORT` subclasses `BYTETracker` |
| Kalman state | `KalmanFilterXYAH` — `x, y, a, h` + velocities | `KalmanFilterXYWH` — `x, y, w, h` + velocities |
| Camera motion handling | ❌ None | ✅ GMC, `gmc_method: sparseOptFlow` by default (`ecc` / `orb` / `none` selectable) |
| Appearance features | ❌ IoU only | ⚠️ Supported, but **`with_reid: False`** by default |
| Cost function | IoU | IoU by default; `α·IoU + (1−α)·ReID` **only** when `with_reid: True` |
| Inference speed | Very fast | Slightly slower (GMC); much slower once ReID is enabled |
| Best for | Static cameras, crowds | Moving cameras; long occlusions *once you turn ReID on* |
| ID switches | More on moving cameras | Fewer on moving cameras; fewer post-occlusion only with ReID |

### The other four trackers

| Tracker | Idea |
|---|---|
| `ocsort.yaml` | Observation-Centric SORT — repairs the Kalman state from observations after an occlusion instead of trusting the error the filter accumulated while blind. |
| `deepocsort.yaml` | OC-SORT plus an appearance embedding. |
| `fasttrack.yaml` | Throughput-oriented association, aimed at edge deployment. |
| `tracktrack.yaml` | TrackTrack (CVPR 2025) — iterative association that keeps more true tracks without admitting more false ones. |

---

## When to use each

**ByteTrack** is the default choice when you need real-time performance on a static camera —
surveillance, sports analytics, retail footfall. It punches far above its weight for the compute
cost, and it is what this notebook benchmarks.

**BoT-SORT** is the right pick when your camera moves (drones, vehicles, handheld) or when
ID-switch errors carry a real cost (sports player tracking, autonomous driving) — remembering that
you must set `with_reid: True` yourself if what you actually need is re-identification after a long
disappearance.

<a name='yolo11'></a>
# 2 - YOLO11 — Instance Segmentation + Multi-Object Tracking

This section benchmarks **Ultralytics YOLO11** (segmentation variant) with **ByteTrack** for
multi-object tracking on video sequences, under the protocol fixed above.

Evaluated metrics:
- **Inference time** per frame — wall clock *and* forward-pass-only (CPU & GPU)
- **FPS** (frames per second)
- **GPU memory** (VRAM) and process RSS
- **Detected masks / active tracks** per frame

> **Not measured here:** mask quality. No mIoU is computed anywhere in this notebook, so
> "SAM 3 trades speed for mask fidelity" stays an assertion — exercise 5 turns it into a number.

> Results are saved to `results/<model>_metrics.json` — one file per model, e.g.
> `results/yolo11n-seg.pt_metrics.json`.

> **Generation note:** YOLO11 is the mature default; **YOLO26** (Jan 2026) is the current
> generation and is natively NMS-free. Several observations below are about NMS and mask
> post-processing cost, i.e. about a stage YOLO26 removes — see exercise 4.

## 2.1 - Core benchmark function

Common YOLO11 input/inference parameters — see the
[Ultralytics predict/config docs](https://docs.ultralytics.com/usage/cfg/):

| Parameter      | Type             | Default        | Description                                                                                         |
| -------------- | ---------------- | -------------- | --------------------------------------------------------------------------------------------------- |
| `source`       | `str`            | —              | Input source: image, video, webcam, folder, URL, RTSP stream, etc.                                  |
| `imgsz`        | `int` or `(h,w)` | `640`          | Input image size. Larger values improve small-object detection but increase latency and VRAM usage. |
| `conf`         | `float`          | `0.25`         | Confidence threshold. Predictions below this score are discarded.                                   |
| `iou`          | `float`          | `0.7`          | IoU threshold for Non-Maximum Suppression (NMS). Controls duplicate box suppression.                |
| `device`       | `str/int`        | `None`         | Execution device: `cpu`, `0`, `0,1`, `cuda:0`, etc.                                                 |
| `quantize`     | `int`            | `32`           | Inference precision: `16` = FP16, `32` = FP32. **Replaces the removed `half` argument**; `half=True` is still accepted as a shim that maps to `quantize=16` and emits a deprecation warning. |
| `batch`        | `int`            | `1`            | Batch size for inference on videos/directories.                                                     |
| `max_det`      | `int`            | `300`          | Maximum detections per image.                                                                       |
| `classes`      | `list[int]`      | `None`         | Filter detections by class IDs.                                                                     |
| `agnostic_nms` | `bool`           | `False`        | Apply class-agnostic NMS.                                                                           |
| `augment`      | `bool`           | `False`        | Test-time augmentation during inference.                                                            |
| `visualize`    | `bool`           | `False`        | Visualize feature maps.                                                                             |
| `show`         | `bool`           | `False`        | Display predictions in a window.                                                                    |
| `save`         | `bool`           | `False`        | Save output predictions.                                                                            |
| `save_txt`     | `bool`           | `False`        | Save detections in YOLO text format.                                                                |
| `save_conf`    | `bool`           | `False`        | Save confidence scores with labels.                                                                 |
| `stream`       | `bool`           | `False`        | Stream inference results instead of returning all at once.                                          |
| `vid_stride`   | `int`            | `1`            | Process every N-th frame in video.                                                                  |
| `tracker`      | `str`            | *version-dependent* | Tracker config. `cfg/default.yaml` on main reads `tracktrack.yaml`; older versions and the docs say `botsort.yaml`. Print `get_cfg().tracker` rather than trusting either. |

Important parameter relationships:

* Lower `conf` → more detections, more false positives.
* Lower `iou` → more aggressive duplicate removal.
* Higher `imgsz` → better accuracy on small objects but slower inference.
* `quantize=16` only helps on GPUs with FP16 / tensor-core support; on CPU Ultralytics falls back
  to FP32, so a "FP16 CPU" number does not exist.

`max_det=300` also matters for this benchmark: it caps how much work NMS and mask upsampling can
be asked to do on a dense frame, which is exactly where the CPU latency tail comes from.

In [ ]:
MODEL_NAMES     = ['yolo11n-seg.pt','yolo11m-seg.pt', 'yolo11x-seg.pt']   # options: yolo11n/s/m/l/x-seg.pt


In [ ]:
def run_yolo11_benchmark(model_name: str, device_label: str, device) -> dict:
    """
    Run YOLO11-seg + ByteTrack on the video and collect per-frame metrics under the shared
    benchmark protocol (fixed imgsz, fixed precision, warm-up, CUDA sync around every timestamp).

    Returns a dict of per-frame lists — `inference_ms` (wall clock: preprocess + forward + NMS +
    mask upsampling + tracker association), `fwd_ms` (forward pass only), `pre_ms`, `post_ms`,
    `n_masks`, `n_tracks`, `gpu_mem_mb`, `cpu_ram_mb` — plus a `summary` that records the protocol
    it was measured under.
    """
    bar = '=' * 60
    print(f'\n{bar}')
    print(f'  YOLO11  |  device={device_label}  |  model={model_name}')
    print(bar)

    reset_gpu_stats()
    model = YOLO(model_name)

    # FP16 has no CPU kernel path in Ultralytics — record the precision that actually ran.
    quantize = QUANTIZE if device != 'cpu' else 32

    track_kwargs = dict(
        tracker=TRACKER,
        device=device,
        imgsz=IMGSZ,
        quantize=quantize,
        conf=CONF_THRESH,
        iou=IOU_THRESH,
        classes=CLASS_FILTER,
        verbose=False,
    )

    metrics = defaultdict(list)
    cap = cv2.VideoCapture(VIDEO_PATH)
    n_frames = MAX_FRAMES or TOTAL_FRAMES

    # optional video writer  (note: model_name, not the loop variable `m`)
    vout = None
    if SAVE_VIDEO:
        out_path = f'output_videos/{model_name}_{device_label.lower()}.mp4'
        vout = cv2.VideoWriter(out_path,
                               cv2.VideoWriter_fourcc(*'mp4v'),
                               FPS_SRC, (W, H))

    # ── warm-up: discarded. Pays for CUDA context creation, cuDNN autotuning and lazy
    #    kernel compilation so they do not land inside the measured frames.
    for _ in range(WARMUP_FRAMES):
        ok, warm_frame = cap.read()
        if not ok:
            break
        _ = model.track(warm_frame, persist=False, **track_kwargs)
    sync()
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    for frame_idx in range(n_frames):
        ret, frame = cap.read()
        if not ret:
            break

        sync()                       # drain anything still queued before the clock starts
        t0 = time.perf_counter()
        results = model.track(frame, persist=True, **track_kwargs)
        sync()                       # ...and wait for the work to finish before it stops
        t1 = time.perf_counter()

        wall_ms = (t1 - t0) * 1000
        r = results[0]

        n_masks  = len(r.masks) if r.masks is not None else 0
        track_ids = (r.boxes.id.cpu().numpy().tolist()
                     if r.boxes.id is not None else [])
        n_tracks = len(set(track_ids))

        # Two scopes, kept apart on purpose (see the Benchmark protocol cell):
        #   inference_ms = wall clock around the whole call, tracking included
        #   fwd_ms       = forward pass only, as Ultralytics reports it
        metrics['inference_ms'].append(wall_ms)
        metrics['fwd_ms'].append(float(r.speed.get('inference', float('nan'))))
        metrics['pre_ms'].append(float(r.speed.get('preprocess', float('nan'))))
        metrics['post_ms'].append(float(r.speed.get('postprocess', float('nan'))))
        metrics['n_masks'].append(n_masks)
        metrics['n_tracks'].append(n_tracks)
        metrics['gpu_mem_mb'].append(get_gpu_memory_mb())
        metrics['cpu_ram_mb'].append(get_cpu_ram_mb())

        if SAVE_VIDEO and vout:
            annotated = r.plot()
            cv2.putText(annotated,
                        f'{device_label}  {wall_ms:.1f}ms  masks:{n_masks}  tracks:{n_tracks}',
                        (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,128), 2)
            vout.write(annotated)

        if frame_idx % 30 == 0:
            print(f'  frame {frame_idx:>4d}/{n_frames}  '
                  f'wall={wall_ms:6.1f}ms  masks={n_masks}  tracks={n_tracks}'
                  f'  vram={get_gpu_memory_mb():.0f}MB')

    cap.release()
    if vout:
        vout.release()

    metrics = dict(metrics)
    arr  = np.array(metrics['inference_ms'], dtype=float)
    fwd  = np.array(metrics['fwd_ms'], dtype=float)
    pre  = np.array(metrics['pre_ms'], dtype=float)
    post = np.array(metrics['post_ms'], dtype=float)
    metrics['summary'] = {
        'mean_ms'      : float(np.mean(arr)),
        'median_ms'    : float(np.median(arr)),
        'std_ms'       : float(np.std(arr)),
        'p95_ms'       : float(np.percentile(arr, 95)),
        'min_ms'       : float(np.min(arr)),
        'max_ms'       : float(np.max(arr)),
        'fps'          : float(1000.0 / np.mean(arr)),
        'mean_fwd_ms'  : float(np.nanmean(fwd)),
        'mean_pre_ms'  : float(np.nanmean(pre)),
        'mean_post_ms' : float(np.nanmean(post)),
        'mean_total_ms': float(np.nanmean(pre + fwd + post)),
        'n_frames'     : len(arr),
        'peak_gpu_mb'  : float(np.max(metrics['gpu_mem_mb'])),
        'peak_ram_mb'  : float(np.max(metrics['cpu_ram_mb'])),
        # protocol provenance — a latency number without these is not reproducible
        'device'       : device_label,
        'imgsz'        : IMGSZ,
        'quantize'     : quantize,
        'tracking'     : TRACKING,
        'warmup_frames': WARMUP_FRAMES,
        'torch_threads': N_THREADS,
    }

    s = metrics['summary']
    print(f'\n  ✔ Done  wall mean={s["mean_ms"]:.1f}ms  forward mean={s["mean_fwd_ms"]:.1f}ms  '
          f'FPS={s["fps"]:.1f}  peak_vram={s["peak_gpu_mb"]:.0f}MB  '
          f'[imgsz={s["imgsz"]} fp{s["quantize"]} threads={s["torch_threads"]}]')
    return metrics

## 2.2 - Run benchmarks

YOLO11 uses the `imgsz` parameter to define the model input resolution.

Typical values:

| `imgsz`        | Use case                          |
| -------------- | --------------------------------- |
| `320`          | Very fast inference, low accuracy |
| `640`          | Default / balanced                |
| `800` / `1024` | Better small-object detection     |
| `1280+`        | High-detail or aerial imagery     |


YOLO11 internally resizes images before inference. For detection models, Ultralytics typically uses **letterboxing** (resize + padding) to preserve aspect ratio.

Important notes:

* Default input size for pretrained YOLO11 models is usually:
  640x640


* Larger `imgsz`:

  * improves small-object detection
  * increases VRAM usage
  * increases latency roughly quadratically

* `imgsz` should usually be a multiple of:
  32
  because YOLO downsamples feature maps by stride 32.

* Non-square inputs are supported:

```python
imgsz=(1280, 736)
```

* During preprocessing:

  * original image is resized
  * padding may be added
  * bounding boxes are automatically remapped to original coordinates

Example for small objects:

```python
results = model.predict(
    source="4k_frame.jpg",
    imgsz=1280
)
```

Example for real-time webcam:

```python
results = model.predict(
    source=0,
    imgsz=416
)
```


In [ ]:
all_results = {}
for m in MODEL_NAMES:

  all_results[m] = {}
  for label, device in DEVICES.items():
      all_results[m][label] = run_yolo11_benchmark(model_name=m,
                                                   device_label=label,
                                                   device=device)

  # Persist results for the comparison notebook — one file per model.
  save_data = {k: v['summary'] for k, v in all_results[m].items()}
  with open(f'results/{m}_metrics.json', 'w') as f:
      json.dump({'model': 'YOLO11-seg', 'config': m,
                'results': save_data}, f, indent=2)

  print(f'\nSaved → results/{m}_metrics.json')

In [ ]:
import cv2
import matplotlib.pyplot as plt

fig , axes = plt.subplots(1,3,figsize=(20,20))

for i,m in enumerate(MODEL_NAMES):
  video_path = f"output_videos/{m}_gpu.mp4"
  cap = cv2.VideoCapture(video_path)
  ret, frame = cap.read()

  if ret:
      frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
      axes[i].imshow(frame)
      axes[i].set_title(m)
      axes[i].axis('off')

  cap.release()


## 2.3 - Per-device plots

### 2.3.1 Inference Time Benchmark

These two graphs compare **per-frame inference latency** across three YOLO11 segmentation model sizes (n, m, x) under two hardware conditions.

In [ ]:
# Fixed palette. The analysis cells below name these colours by hand, so they must not change
# from run to run — an unseeded random palette can also land on near-white or on two identical hues.
PALETTE = {'CPU': '#4C9BE8', 'GPU': '#F2A541'}
PALETTE_BY_MODEL = {
    'yolo11n-seg.pt': '#1f77b4',   # blue
    'yolo11m-seg.pt': '#2ca02c',   # green
    'yolo11x-seg.pt': '#9467bd',   # purple
    'sam3'          : '#d62728',   # red
}
_FALLBACK = ['#8c564b', '#17becf', '#e377c2', '#7f7f7f', '#bcbd22']
for _i, _m in enumerate(MODEL_NAMES):
    PALETTE_BY_MODEL.setdefault(_m, _FALLBACK[_i % len(_FALLBACK)])


def smooth(x, w=7):
    return np.convolve(x, np.ones(w)/w, mode='valid')

n_devices = len(DEVICES)
fig, axes = plt.subplots(n_devices, 1,
                          figsize=(14, 4.5 * n_devices), sharex=False)
if n_devices == 1:
    axes = [axes]

for m in MODEL_NAMES:
  # ── 1. Inference time over frames ──────────────────────────────

  metrics = all_results[m]

  model_name_base = m.split(".")[0].split("-")[0]
  for ax, (label, data) in zip(axes, metrics.items()):
      ms   = np.array(data['inference_ms'])
      xs   = np.arange(len(ms))
      col  = PALETTE_BY_MODEL.get(m, 'steelblue')
      ax.fill_between(xs, ms, alpha=0.15, color=col)
      ax.plot(xs, ms, color=col, alpha=0.4, lw=0.8, label=f'{model_name_base} raw')
      ax.plot(np.arange(len(smooth(ms))), smooth(ms),
              color=col, lw=2)
      ax.axhline(data['summary']['mean_ms'], ls='--', color=col,
                lw=1.4, label=f'{model_name_base} mean = {data["summary"]["mean_ms"]:.1f} ms')
      ax.set_title(f'Yolo11-seg  |  {label}  —  Inference Time per Frame (wall clock, tracking included)',
                   fontsize=13, fontweight='bold')
      ax.set_ylabel('Latency (ms)')
      ax.set_xlabel('Frame index')
      ax.legend()
      ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/yolo11_inference_time.png', dpi=150, bbox_inches='tight')
plt.show()

#### Analysis

> ⚠️ **Reference-run numbers.** The latencies quoted in this cell are from one run on a Colab T4
> over 200 frames of `shinjuku.mp4`, at `imgsz=640`, and are wall clock *including* tracking. They
> are kept so the discussion has something concrete to point at. Your run will differ — the figure
> above and the tables generated further down are the current truth.

---

### Top graph — CPU

| Model | Mean latency (reference run) |
|---|---|
| yolo11n | 135.0 ms |
| yolo11m | 730.2 ms |
| yolo11x | 1842.5 ms |

- **yolo11n** is fast and stable on CPU (~135 ms ≈ ~7 FPS)
- **yolo11m** is ~5× slower than n
- **yolo11x** is extremely slow (~1.8 s/frame), making it **unusable for real-time CPU inference**
- The large **shaded bands** around yolo11x indicate high variance — latency spikes heavily (up to ~4500 ms) around frames 130–160, likely due to complex scenes with many objects/masks to process
- yolo11n has almost no visible band → very consistent timing

---

### Bottom graph — GPU

| Model | Mean latency (reference run) |
|---|---|
| yolo11n | 17.9 ms (~56 FPS) |
| yolo11m | 23.4 ms (~43 FPS) |
| yolo11x | 45.7 ms (~22 FPS) |

- **All three models are viable for real-time** on GPU
- The gap between n and x is only ~2.5× (vs ~13× on CPU) — GPU parallelism dramatically narrows the difference
- There's a **large spike around frame ~110–115** across all models simultaneously → this is almost certainly a scene with unusually high object density or large masks, not a model artifact
- After frame ~130, yolo11x stabilizes back to its mean

---

### Key takeaways

- **GPU is non-negotiable for yolo11m/x** in real-time applications
- **yolo11n on GPU** is the sweet spot for edge deployment (~18 ms, stable)
- The correlated spike at frame ~110 across all models tells you the bottleneck there is **the input data** (hard frame), not the model itself
- CPU variance in yolo11x suggests it is sensitive to scene complexity. The plausible driver is the
  part of the pipeline whose cost scales with *detection count* rather than image size: NMS and
  mask upsampling. Note this is a hypothesis about this curve, not a measurement — the wall-clock
  number bundles those stages together with the forward pass. `mean_pre_ms` / `mean_fwd_ms` /
  `mean_post_ms` are recorded per run precisely so you can separate them, and **YOLO26 removes the
  NMS stage entirely**, which makes it the cleanest available test (exercise 4).

### 2.3.2 Detected Masks & Active Tracks per Frame


This is a 2×2 grid comparing **what each model detects**, not how fast it runs. Hardware (CPU/GPU) is in the rows, metric type in the columns.



In [ ]:
# ── 2. Masks & tracks per frame ────────────────────────────────
fig, axes = plt.subplots(2, 2,
                          figsize=(16, 4 * len(all_results)))
if len(all_results) == 1:
    axes = [axes]

for m in MODEL_NAMES:
  metrics = all_results[m]

  model_name_base = m.split(".")[0].split("-")[0]

  for row, (label, data) in enumerate(metrics.items()):
      col = PALETTE_BY_MODEL.get(m, 'steelblue')
      xs  = np.arange(len(data['n_masks']))

      for ax, key, title in zip(
          axes[row],
          ['n_masks', 'n_tracks'],
          ['Detected Masks / Frame', 'Active Tracks / Frame'],
      ):
          vals = np.array(data[key])
          ax.step(xs, vals, where='mid', color=col, lw=1.5)
          ax.fill_between(xs, vals, step='mid', alpha=0.15, color=col)
          ax.axhline(np.mean(vals), ls='--', color=col, lw=1.2,
                    label=f'{model_name_base} mean = {np.mean(vals):.1f}')
          ax.set_title(f'YOLO11 | {label} — {title}', fontweight='bold')
          ax.set_xlabel('Frame')
          ax.legend()
          ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/yolo11_masks_tracks.png', dpi=150, bbox_inches='tight')
plt.show()

#### Analysis
---

> ⚠️ **Reference-run numbers**, as everywhere in this notebook's prose. The colours, however, are
> now fixed by `PALETTE_BY_MODEL`, so "blue = yolo11n" is stable across runs.

### What each plot shows

| Column | Metric |
|---|---|
| Left | **Detected Masks/Frame** — raw segmentation output count per frame |
| Right | **Active Tracks/Frame** — objects being tracked (post-tracker filtering) |

---

### The most important observation

**The CPU and GPU plots are nearly identical** — the curves are virtually the same across all 4 panels. This confirms that hardware doesn't affect detection results, only speed. The model weights determine what gets detected.

(Strictly: *nearly* identical, not bit-identical. The GPU runs at FP16 and the CPU at FP32, so a
handful of detections sitting right on the `conf=0.30` boundary can flip. If you ever see a large
difference here, that is a bug, not a hardware effect.)

---

### Per-model behavior (reference run)

**yolo11n (blue) — mean 3.7**
- Detects very few objects per frame, consistently low
- Effectively conservative — it only fires on obvious, high-confidence detections
- Very stable, little variance

**yolo11m (green) — mean 16.0**
- Detects the *most* on average, even more than `x`
- This is counterintuitive at first glance

**yolo11x (purple) — mean 13.9**
- Detects significantly more objects than `n`, but slightly fewer than `m`

---

### Why does yolo11m detect more than yolo11x?

This is the most interesting finding in these graphs. Possible explanations:

- **yolo11x is more selective** — it has higher effective precision, so it suppresses borderline detections that `m` passes through
- **Calibration difference** — larger models often produce better-calibrated confidences, so fewer
  marginal objects clear a fixed `conf=0.30`

This does **not** mean `m` is better than `x` — it may mean `x` is more precise (fewer false positives), while `m` casts a wider net. Counting detections is not evaluating them: without ground
truth you cannot tell a recovered small object from a false positive. That is what exercise 5 is for.

---

### Detected Masks vs. Active Tracks (left vs. right)

The two columns are almost identical in shape, which means:
- The tracker is keeping up with detections — very little track loss
- Minimal ID-switch noise visible
- The tracker isn't adding or suppressing much relative to raw detections

If there were significant differences between the two columns, it would indicate tracker instability (lost tracks, ghost tracks, etc.).

---

### Scene structure visible in the signal

The detection counts follow a clear pattern across all models:
- **Frames 0–50**: lower count, simpler scene
- **Frames 75–100**: scene gets busier, spike in detections
- **Frames 150–200**: peak complexity, highest counts (up to 25+ objects)

This matches the latency spike you saw at ~frame 110 in the previous graphs — dense scenes drive both higher latency and higher detection counts simultaneously.

### 2.3.3 CPU vs GPU summary bar chart
Three metrics side by side, all three models, both hardware backends.


In [ ]:
# ── 3. CPU vs GPU summary bar chart (if both available) ────────
# Grouped bars: every model gets its own x offset. Drawing all three models at the same two
# categorical x positions would paint each over the previous one and only the tallest would show.
if len(all_results[MODEL_NAMES[0]]) > 1:
    metrics_keys = ['mean_ms', 'p95_ms', 'fps']
    titles   = ['Mean latency (ms)', 'P95 latency (ms)', 'FPS (1000/mean_ms)']
    labels   = list(DEVICES.keys())          # x categories: CPU, GPU
    x        = np.arange(len(labels))
    width    = 0.8 / len(MODEL_NAMES)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, key, title in zip(axes, metrics_keys, titles):
        for i, m in enumerate(MODEL_NAMES):
            model_base_name = m.split('.')[0].split('-')[0]
            vals   = [all_results[m][l]['summary'][key] for l in labels]
            offset = (i - (len(MODEL_NAMES) - 1) / 2) * width
            bars = ax.bar(x + offset, vals, width=width,
                          color=PALETTE_BY_MODEL.get(m, 'steelblue'),
                          edgecolor='white', linewidth=1.2, label=model_base_name)
            ax.bar_label(bars, fmt='%.1f', padding=2, fontsize=8, rotation=90)

        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_title(title, fontsize=12, fontweight='bold')
        if key != 'fps':
            # CPU and GPU latencies differ by ~2 orders of magnitude: on a linear axis the GPU
            # bars are invisible, which is a plotting artefact rather than a finding.
            ax.set_yscale('log')
            ax.set_ylabel('ms (log scale)')
        ax.margins(y=0.25)
        ax.grid(axis='y', alpha=0.3)
        ax.spines[['top','right']].set_visible(False)

    axes[0].legend(frameon=False, fontsize=9)
    fig.suptitle('YOLO11-seg  |  CPU vs GPU Summary (wall clock)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('results/yolo11_cpu_vs_gpu.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Only one device available — skipping CPU vs GPU comparison plot.')

In [ ]:
def remove_outliers_percentile(arr, low=5, high=95):
    """Clip to the [low, high] percentile band. NOTE: this deletes the latency tail — the part
    most of this notebook's analysis is about. Only use it to see the bulk of the distribution."""
    lo, hi = np.percentile(arr, [low, high])
    return arr[(arr >= lo) & (arr <= hi)]


def box_plot(data: dict, remove_outliers: bool = True, model_name: str = None,
             out_path: str = None):
  # ── 4. Latency distribution (box plot + strip) ─────────────────
  fig, ax = plt.subplots(figsize=(8, 5))

  if remove_outliers:
    data_list  = [remove_outliers_percentile(np.array(data[l]['inference_ms'])) for l in data]
    scope = '5th–95th percentile — tails clipped'
  else:
    data_list  = [np.array(data[l]['inference_ms']) for l in data]
    scope = 'all frames'

  colors_box = [PALETTE.get(l, 'steelblue') for l in data]

  bp = ax.boxplot(data_list, patch_artist=True, notch=True,
                  medianprops=dict(color='white', linewidth=2))
  for patch, col in zip(bp['boxes'], colors_box):
      patch.set_facecolor(col)
      patch.set_alpha(0.7)

  # jitter overlay
  for i, (data_arr, col) in enumerate(zip(data_list, colors_box), start=1):
      jitter = np.random.default_rng(0).normal(0, 0.07, len(data_arr))
      ax.scatter(i + jitter, data_arr, alpha=0.25, s=12, color=col)

  ax.set_xticks(np.arange(1, len(data) + 1))
  ax.set_xticklabels(list(data.keys()), fontsize=12)
  ax.set_ylabel('Latency (ms)', fontsize=11)
  ax.set_title(f'{model_name}  |  Latency Distribution ({scope})', fontsize=13, fontweight='bold')
  ax.grid(axis='y', alpha=0.3)
  plt.tight_layout()

  # One file per model and per scope — a single hardcoded path made the loop overwrite itself.
  if out_path is None:
      stem = (model_name or 'model').split('.')[0]
      suffix = 'clipped' if remove_outliers else 'full'
      out_path = f'results/{stem}_latency_dist_{suffix}.png'
  plt.savefig(out_path, dpi=150, bbox_inches='tight')
  plt.show()

In [ ]:
for m in MODEL_NAMES:
  # Both scopes: the clipped view shows the bulk, the full view shows the tail we care about.
  box_plot(all_results[m], remove_outliers=True,  model_name=m)
  box_plot(all_results[m], remove_outliers=False, model_name=m)

In [ ]:
import pandas as pd

SUMMARY_KEYS = ['mean_ms', 'median_ms', 'std_ms', 'p95_ms', 'fps',
                'mean_fwd_ms', 'mean_total_ms', 'peak_gpu_mb', 'peak_ram_mb']


def summary_frame(results: dict, keys=SUMMARY_KEYS) -> pd.DataFrame:
    """
    Build the comparison table straight from the measured runs.

    Every latency table in the analysis cells below is produced by this function. Do not retype
    these numbers into markdown: hand-copied figures go stale the first time anyone re-runs the
    notebook, and a mistyped P95 can end up smaller than its own mean.
    """
    rows = []
    for model, per_device in results.items():
        for device, run in per_device.items():
            s = run['summary']
            row = {'Model': model.split('.')[0], 'Device': device}
            row.update({k: (round(float(s[k]), 2) if k in s else np.nan) for k in keys})
            row['imgsz']    = s.get('imgsz', '?')
            row['fp']       = s.get('quantize', '?')
            row['threads']  = s.get('torch_threads', '?')
            rows.append(row)
    df = pd.DataFrame(rows).set_index(['Model', 'Device'])
    df['p95/mean'] = (df['p95_ms'] / df['mean_ms']).round(2)
    return df


summary_df = summary_frame(all_results)
print('Latency summary — wall clock (mean_ms) vs forward-pass only (mean_fwd_ms)')
print(summary_df.to_string())

print('\nCPU→GPU speedup on mean wall clock:')
for m in MODEL_NAMES:
    if 'CPU' in all_results[m] and 'GPU' in all_results[m]:
        cpu = all_results[m]['CPU']['summary']['mean_ms']
        gpu = all_results[m]['GPU']['summary']['mean_ms']
        print(f'  {m.split(".")[0]:<14} {cpu/gpu:6.1f}x')

#### Analysis

> ⚠️ **Read the table printed by the cell above, not numbers typed into this text.** Earlier
> versions of this notebook hardcoded a P95 table here, and one of the values (a GPU P95 smaller
> than half its own mean) was arithmetically impossible — a right-skewed latency distribution can
> never have `P95 < mean`. That is precisely the class of error that generating the table removes.
> The prose below describes *how to read* the table; it deliberately quotes as few numbers as
> possible, and the ones it does quote are from the reference run (Colab T4, 200 frames).

---

### Mean latency

Read the `mean_ms` column, CPU rows against GPU rows, and the printed CPU→GPU speedup underneath.
The pattern to look for: the speedup **grows with model size**. Small models on CPU are partly
bound by per-frame overhead that the GPU cannot remove, while large models are bound by the
convolutional work the GPU is built for. In the reference run this ran from roughly 7× for
yolo11n to roughly 40× for yolo11x.

Also compare `mean_ms` against `mean_fwd_ms` in the same row. The difference is everything the
wall clock includes and the forward pass does not: preprocessing, NMS, mask upsampling and the
ByteTrack association. On CPU that gap is large; on GPU it can be most of the frame budget.

The latency panels are on a **log axis** — on a linear axis the GPU bars sit at a couple of pixels
next to a 1.8-second CPU bar, and "the GPU bars are nearly invisible" is a statement about
matplotlib, not about hardware.

---

### P95 latency and the tail

The `p95/mean` column is the one to read. A ratio near 1.0 means latency is predictable
frame-to-frame; a ratio well above 1.0 means a heavy tail, i.e. some frames cost far more than the
average and your pipeline needs to be sized for them, not for the mean.

Two questions worth answering from your own run:

1. Do the CPU rows show a heavier tail than the GPU rows for the same model? (In the reference
   run they did — CPU tails were driven by dense frames, where NMS and mask upsampling scale with
   detection count.)
2. Does the tail ratio get better or worse with model size? A bigger model is slower everywhere,
   which can make its *relative* tail look smaller even though its absolute worst frame is worse.

Note the box plots above are drawn twice on purpose — the clipped version shows the bulk of the
distribution, the full version shows the tail. The clipped one is the prettier chart and the wrong
one to reason about tails from.

---

### FPS

`fps` is just `1000 / mean_ms`, so it carries no information the latency columns do not — it is
there because deployment budgets are written in FPS. Two reading rules: mean-based FPS says
nothing about whether you will drop frames (that is the P95 column's job), and an FPS below ~1
means seconds per frame, i.e. an offline batch job, not a video pipeline.

---

### Bottom line (reference run)

| Decision | Recommendation |
|---|---|
| Real-time, edge CPU | yolo11n only — and check its tail before you promise a frame rate |
| Real-time, GPU | Any of the three; pick by accuracy, not by latency |
| Latency-critical prod | GPU + the smallest model that meets your accuracy bar, chosen on P95 rather than mean |
| Best accuracy/speed tradeoff on GPU | yolo11m — clearly better detection than n, well under the frame budget |

### 2.3.4 Summary

> ⚠️ Reference-run figures (Colab T4, 200 frames, `imgsz=640`). The generated table above is the
> authority for your run.

**Latency tails (mean vs P95 on CPU)**

| Model | Mean | P95 | Tail ratio |
|---|---|---|---|
| yolo11n | 135 ms | 261 ms | 1.93× |
| yolo11m | 730 ms | 984 ms | 1.35× |
| yolo11x | 1842 ms | 2731 ms | 1.48× |

yolo11n has the worst tail ratio — its mean is pulled down by fast frames, but complex scenes hit it hard proportionally. yolo11x is already slow everywhere so the tail ratio looks smaller. The
`p95/mean` column in the generated table computes exactly this, so you can check whether the
ordering survives on your hardware.

**GPU memory is the real constraint for x**
---

In the reference run yolo11x on GPU peaked at 385 MB VRAM — fine for any modern discrete GPU, but tight on integrated graphics or older mobile GPUs. yolo11n at 50 MB VRAM is virtually free. Note
that `peak_gpu_mb` uses `torch.cuda.memory_allocated()`, so it counts tensors, not the CUDA context
and cuDNN workspaces — the real process footprint is several hundred MB larger.

**Std dev tells a different story than mean**
---

On CPU, yolo11n std = 98.9 ms against a mean of 135 ms — a coefficient of variation of ~73%. The model is fast on average but very unpredictable frame-to-frame. yolo11m by comparison had a CV of ~17%, much more consistent. If you need predictable latency (e.g. pipeline buffering), read `std_ms`
and `p95/mean` together rather than the mean alone.

In [ ]:
for m in MODEL_NAMES:
  rows = []
  for label, data in all_results[m].items():
      s = data['summary']
      rows.append({
          'Device'      : label,
          'Mean ms'     : round(s['mean_ms'], 2),       # wall clock, tracking included
          'Fwd ms'      : round(s['mean_fwd_ms'], 2),   # forward pass only
          'Median ms'   : round(s['median_ms'], 2),
          'Std ms'      : round(s['std_ms'], 2),
          'P95 ms'      : round(s['p95_ms'], 2),
          'Min ms'      : round(s['min_ms'], 2),
          'Max ms'      : round(s['max_ms'], 2),
          'FPS'         : round(s['fps'], 1),
          'Peak VRAM MB': round(s['peak_gpu_mb'], 1),
          'Peak RSS MB' : round(s['peak_ram_mb'], 1),
          'imgsz'       : s['imgsz'],
          'fp'          : s['quantize'],
      })

  print("="*50)
  print(m)

  df = pd.DataFrame(rows).set_index('Device')
  print(df.to_string())

<a name='sam3'></a>
# 3 - SAM 3

SAM 3's defining task is **Promptable Concept Segmentation (PCS)**: you give it a *concept* — a
short noun phrase, an image exemplar, or both — and it segments and tracks **every instance of that
concept** in an image or video. It is open-vocabulary, but it is **not** class-agnostic and there is
no "segment everything" mode in play here: further down we prompt it with `text=["person", "car"]`
at `conf=0.5`, so every mask it returns is a person or a car.

Input resolution
---

The native input size for SAM 3 is **1008×1008** pixels.

- **Standard implementation**: optimized for 1008×1008 px.
- **Divisibility**: the size must be divisible by the patch size, which is **14** — 1008/14 = 72, so
  the image becomes a 72×72 grid of patch tokens (5184 tokens).
- **Square aspect ratio**: images are letterboxed/resized to a square.
- **Pretraining resolution**: the Perception Encoder backbone initializes its position embeddings
  from a 336×336 pretraining resolution and interpolates them up.
- **Custom resolutions**: allowed, but expect mask quality to degrade away from the intended size.
- **Automatic resizing**: `Sam3Processor` on Hugging Face resizes the input and post-processes masks
  back to the original dimensions.

> **What we do here, and why:** Ultralytics' own `SAM3VideoSemanticPredictor` example runs at
> `imgsz=640`, and our benchmark protocol requires both models to see the same input resolution, so
> we pass `imgsz=IMGSZ` (640) explicitly. That is a deliberate, recorded protocol choice — not the
> model's native resolution. Exercise 6 asks you to re-run at 1008 and measure what it costs in
> latency and in mask quality.

### Getting the weights — `facebook/sam3` is a **gated** repo

Two things about `sam3.pt` that will otherwise waste half an hour of class time:

- Ultralytics does **not** auto-download it the way it downloads `yolo11n-seg.pt`. You fetch it
  yourself from the Hub.
- `facebook/sam3` is gated with **manual** approval (`gated: "manual"`). Even `config.json` returns
  **401** without a token, so an unauthenticated `hf_hub_download` raises `GatedRepoError` — and
  the traceback looks like the token *was* used, which is the confusing part.

Before running the next cell:

1. Open <https://huggingface.co/facebook/sam3> while logged in and click **Request access**. Accept
   the license terms. Approval is manual and **not** instantaneous — do this before the class, not
   during it.
2. Create an access token at <https://huggingface.co/settings/tokens>. A **read** token is enough.
3. In Colab, open the **🔑 Secrets** panel in the left sidebar, add a secret named exactly
   `HF_TOKEN` with that value, and switch on **Notebook access** for this notebook.

The next cell then passes that token to `hf_hub_download(..., token=HF_TOKEN)` — reading the secret
without passing it does nothing at all.

In [ ]:
from huggingface_hub import hf_hub_download
from google.colab import userdata

REPO_ID  = "facebook/sam3"
FILENAME = "sam3.pt"

HF_TOKEN = userdata.get('HF_TOKEN')      # Colab secret — see the cell above
if not HF_TOKEN:
    raise RuntimeError(
        "No HF_TOKEN found. Add it under Colab → Secrets (🔑) and enable notebook access. "
        "facebook/sam3 is a gated repo: the download fails with 401 / GatedRepoError without "
        "an approved token."
    )

# The token has to be *passed*. Merely calling userdata.get() authenticates nothing.
sam3_weights = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    local_dir="/content/",
    token=HF_TOKEN,
)
print('SAM 3 weights →', sam3_weights)

In [ ]:
from ultralytics.models.sam import SAM3VideoSemanticPredictor

# Same protocol as the YOLO11 runs: same input size, same precision, temporal association on.
overrides = dict(
    conf=0.5,
    task="segment",
    mode="predict",
    model=sam3_weights,
    imgsz=IMGSZ,          # explicit: SAM 3 defaults to 1008, YOLO11 to 640 — see the protocol cell
    quantize=QUANTIZE,    # replaces the deprecated `half=True` shim
    save=True,
)

try:
    del predictor
except NameError:
    pass

predictor = SAM3VideoSemanticPredictor(overrides=overrides)

## 3.1 Core benchmark functions

### 3.1.1 Defining hooks to get metrics per frame

In [ ]:
import time
import psutil
import numpy as np
import torch
import cv2
from typing import Literal
from collections import defaultdict


class ResourceMonitor:
    """
    Per-frame metric collector for an Ultralytics predictor.

    Despite what the name suggests, this samples nothing in the background: there is no thread and
    no timer. Every list below is appended exactly once per frame from the `on_predict_batch_end`
    callback registered further down, so the sampling rate is 'one sample per frame'.
    """

    def __init__(self, device: Literal["cpu", "gpu"]):
      self.device = device
      self.pre_process_speed = []
      self.inference_speed = []
      self.post_process_speed = []
      self.ram_mb = []                # process RSS in MB. This is *memory*, not CPU percent.
      self.classes_by_name = []
      self.n_instances_by_class = []
      self.vram_mb = []
      self.n_mask = []
      self.n_tracks = []
      self.peak_gpu_mb = []           # running maxima, one entry per frame
      self.peak_ram_mb = []
      self.n_seen = 0                 # frames observed, including discarded warm-up frames

    def summary(self):
      fwd   = np.asarray(self.inference_speed, dtype=float)
      pre   = np.asarray(self.pre_process_speed, dtype=float)
      post  = np.asarray(self.post_process_speed, dtype=float)
      total = pre + fwd + post        # element-wise: one total per frame

      return {
          'instances_last_frame': self.n_instances_by_class[-1],
          'classes_last_frame': self.classes_by_name[-1],
          # `mean_ms` here is the FORWARD PASS ONLY, the same scope as `mean_fwd_ms` on the YOLO
          # side. `mean_total_ms` adds preprocessing and postprocessing. Compare like with like.
          'mean_ms': float(np.mean(fwd)),
          'median_ms': float(np.median(fwd)),
          'std_ms': float(np.std(fwd)),
          'p95_ms': float(np.percentile(fwd, 95)),
          'min_ms': float(np.min(fwd)),
          'max_ms': float(np.max(fwd)),
          'fps': float(1000.0 / np.mean(fwd)),
          'mean_fwd_ms': float(np.mean(fwd)),
          'mean_pre_ms': float(np.mean(pre)),
          'mean_post_ms': float(np.mean(post)),
          'mean_total_ms': float(np.mean(total)),
          'n_frames': len(fwd),
          'peak_gpu_mb': float(np.max(self.vram_mb)),
          'peak_ram_mb': float(np.max(self.ram_mb)),
          # protocol provenance, mirroring the YOLO summaries
          'device': 'GPU' if self.device == 'gpu' else 'CPU',
          'imgsz': IMGSZ,
          'quantize': QUANTIZE,
          'tracking': TRACKING,
          'warmup_frames': WARMUP_FRAMES,
          'torch_threads': N_THREADS,
      }


def make_yolo_hooks(monitor: ResourceMonitor):
    """
    Returns a post-batch callback to register on an Ultralytics predictor.
    Captures per-frame speed and resource usage, discarding the first WARMUP_FRAMES.
    """

    def post_hook(predictor):
      sync()                       # CUDA is async: make sure the frame really finished
      monitor.n_seen += 1
      if monitor.n_seen <= WARMUP_FRAMES:
        return                     # warm-up frame: same discard rule as the YOLO benchmark

      results = predictor.results[0]

      n_instances_by_class = {}
      for box, clsid in zip(results.boxes.xyxy, results.boxes.cls):
        clsid = int(clsid)
        if clsid not in n_instances_by_class:
          n_instances_by_class[clsid] = 0
        n_instances_by_class[clsid] += 1

      monitor.inference_speed.append(results.speed["inference"])
      monitor.pre_process_speed.append(results.speed["preprocess"])
      monitor.post_process_speed.append(results.speed["postprocess"])
      monitor.n_mask.append(len(results.masks) if results.masks is not None else 0)
      monitor.classes_by_name.append(results.names)
      monitor.n_instances_by_class.append(n_instances_by_class)
      monitor.n_tracks.append(len(results.boxes))
      monitor.ram_mb.append(get_cpu_ram_mb())
      monitor.vram_mb.append(get_gpu_memory_mb())
      monitor.peak_gpu_mb.append(float(np.max(monitor.vram_mb)))
      monitor.peak_ram_mb.append(float(np.max(monitor.ram_mb)))
    return post_hook

In [ ]:
monitor = ResourceMonitor('gpu' if HAS_CUDA else 'cpu')
post_hook = make_yolo_hooks(monitor)
predictor.add_callback("on_predict_batch_end", post_hook)

# Promptable Concept Segmentation: SAM 3 only returns instances of the concepts we ask for.
# Two concepts, one confidence threshold — this is NOT an "everything" mode.
results = predictor(
    source=VIDEO_PATH,
    text=["person", "car"],
    stream=True
)

### 3.1.2 Inference

In [ ]:
# `stream=True` returns a lazy generator: nothing runs until we iterate.
# The monitor discards the first WARMUP_FRAMES, so consume warm-up + measured frames.
for i, r in enumerate(results):
    if i >= WARMUP_FRAMES + MAX_FRAMES:
      break

print(f'frames observed: {monitor.n_seen}  |  frames recorded: {len(monitor.inference_speed)}')

## 3.2 Inference time analysis

> ⚠️ Reference-run figures (Colab T4). The plots and tables below are regenerated from `monitor`
> and `all_results`; this prose is not.

In the reference run SAM 3 averaged **1490.7 ms per frame on GPU**, against 45.7 ms for
yolo11x-seg on the same GPU — **≈33× on this protocol**.

Read that "≈33×" carefully: it is a property of *this measurement*, not of the two models. Same
video, same 200 frames, same T4, same input resolution and precision, forward-pass scope on both
sides. Change any of those and the ratio moves — which is exactly why the protocol cell exists, and
why the ratio is recomputed from the measured summaries in section 4 rather than quoted.

What the number does support is the order of magnitude, and that is the lesson: SAM 3 is a
fundamentally heavier class of model, not a slower YOLO.

In [ ]:
# ── Assemble the SAM 3 run into the same shape as an all_results[...] entry ──
inf_ms  = np.asarray(monitor.inference_speed, dtype=float)
pre_ms  = np.asarray(monitor.pre_process_speed, dtype=float)
post_ms = np.asarray(monitor.post_process_speed, dtype=float)

# Element-wise per frame. `np.sum([a, b, c])` would collapse three 200-element lists to a single
# scalar, and `np.mean([a, b, c])` averaged over all 600 values at once — which is why the old
# `mean_total` came out at roughly a third of the true mean total latency.
total_ms = pre_ms + inf_ms + post_ms

data = {
    'inference_ms': monitor.inference_speed,      # forward pass only
    'pre_process_ms': monitor.pre_process_speed,
    'post_process_ms': monitor.post_process_speed,
    'total_ms': total_ms,                         # one total per frame
    'mean_total': float(np.mean(total_ms)),       # mean of the per-frame totals
    'ram_mb': monitor.ram_mb,                     # process RSS in MB (not CPU %)
    'vram_mb': monitor.vram_mb,
    'n_masks': monitor.n_mask,
    'n_tracks': monitor.n_tracks,
    'classes_by_name': monitor.classes_by_name,
    'n_instances_by_class': monitor.n_instances_by_class,
    'summary': monitor.summary()
}

print(f"SAM 3  |  frames={len(inf_ms)}  "
      f"forward mean={np.mean(inf_ms):.1f} ms  "
      f"pre={np.mean(pre_ms):.1f} ms  post={np.mean(post_ms):.1f} ms  "
      f"total mean={data['mean_total']:.1f} ms")


def smooth(x, w=7):
    return np.convolve(x, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(1, 1, figsize=(14, 4.5), sharex=False)

label = "GPU" if HAS_CUDA else "CPU"
model_name_base = "SAM 3"

ms   = np.array(data['inference_ms'])
xs   = np.arange(len(ms))
col  = PALETTE_BY_MODEL['sam3']
ax.fill_between(xs, ms, alpha=0.15, color=col)
ax.plot(xs, ms, color=col, alpha=0.4, lw=0.8, label=f'{model_name_base} raw')
ax.plot(np.arange(len(smooth(ms))), smooth(ms),
        color=col, lw=2)
ax.axhline(data['summary']['mean_ms'], ls='--', color=col,
          lw=1.4, label=f'{model_name_base} mean = {data["summary"]["mean_ms"]:.1f} ms')
ax.set_title(f'SAM 3 |  {label}  —  Inference Time per Frame (forward pass only)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Latency (ms)')
ax.set_xlabel('Frame index')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/sam3_inference_time.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2.1 GPU — Inference Time per Frame

> ⚠️ Reference-run figures (Colab T4, 200 frames). Frame ranges are specific to `shinjuku.mp4`.

Latency profile analysis
---

**Range:** ~1000–2600 ms, with most frames between 1100–1700 ms.

The curve has a clear **wave structure** — not random noise, but correlated with scene content:

| Frame range | Latency | Interpretation |
|---|---|---|
| 0–5 | ~1000 ms dip | Simple first frames (the true warm-up frames are discarded before recording) |
| 5–25 | ~2100 ms peak | Dense/complex scene |
| 25–75 | Declining 2100→1100 ms | Scene simplifies |
| 75–125 | Trough ~1050–1150 ms | Simplest frames in video |
| 125–135 | Spike ~1600 ms | Local complexity burst |
| 135–185 | ~1200–1600 ms | Mid-complexity |
| 185–195 | Peak ~2100–2600 ms | Hardest frames in video |


Why SAM 3 is this slow
---

SAM 3 is **not** SAM 1/2 with a video loop bolted on, and its cost is not "a prompt encoder plus an
image encoder plus a mask decoder, all per frame". That was the SAM 1/2 design. SAM 3 is a single
**shared backbone feeding two heads**, and the cost comes from three places:

1. **A large shared backbone.** A ~450M-parameter **Perception Encoder (PE) ViT** runs once per
   frame at **1008×1008** natively with **patch size 14** → a 72×72 grid = **5184 tokens** (not
   1008² tokens, and not one token per pixel). Its attention is **windowed** (`window_size: 24`),
   with *global* attention only at layers 7, 15, 23 and 31 — so 28 of its 32 layers are local. The
   backbone is expensive because it is big and runs over thousands of tokens, not because every
   layer does global self-attention over the whole image.
2. **A DETR-style detector on top.** The backbone feeds a DETR decoder over learned object queries,
   with a **presence head** that decouples recognition ("is this concept in the frame at all?") from
   localization ("where is it?"). That decoder is shared with...
3. **...a memory-based tracker.** Per frame, memory attention over a bank of previous frames keeps
   masklets consistent over time. YOLO11 + ByteTrack pays nothing comparable — ByteTrack's
   association is a Hungarian solve on a small IoU matrix, microseconds of CPU work.

YOLO11-seg, for contrast, is one convolutional forward pass at 640² plus a linear combination of 32
learned mask prototypes. Different amount of computation, different kind of answer.

---

### The shaded band is narrow

Unlike YOLO11x on CPU (huge variance), the raw signal stays close to the smoothed line — the shaded region is thin. This means **SAM 3 latency is consistent** — it is always slow, but predictably so. The variance is driven by scene content (how many instances the DETR decoder and the mask head have
to resolve), not by model instability.

---

### Practical bottom line

| Use case | Verdict |
|---|---|
| Real-time video (≥15 FPS) | Not viable with SAM 3 on this hardware |
| Offline batch segmentation | Fine — 1–2 s/frame is acceptable |
| High-quality single-image masking | Ideal use case |
| Open-vocabulary concepts YOLO's COCO head cannot name | The reason to reach for SAM 3 at all |
| Combined with YOLO (detect → SAM refine) | Possible offline, not real-time |

---

**YOLO11 buys speed with prototype-based masks and a fixed 80-class vocabulary; SAM 3 buys
open-vocabulary, promptable, temporally consistent masks and pays for them in latency. They solve
different problems.**

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

xs = np.arange(len(data['n_masks']))

# Fixed, named colours — the analysis text below refers to them.
PALETTE_BY_CLASS = {
    "person": "#d62728",   # red
    "car"   : "#1f77b4",   # blue
}

# Loop for n_masks and n_tracks plots
for ax, key, title in zip(
    axes,
    ['n_masks', 'n_tracks'],
    ['Detected Masks / Frame', 'Active Tracks / Frame'],
):
    vals = np.array(data[key])
    ax.step(xs, vals, where='mid', color=col, lw=1.5)
    ax.fill_between(xs, vals, step='mid', alpha=0.15, color=col)
    ax.axhline(np.mean(vals), ls='--', color=col, lw=1.2,
              label=f'SAM 3 mean = {np.mean(vals):.1f}')

    ax.set_title(f'SAM 3 | {label} — {title}', fontweight='bold')
    ax.set_xlabel('Frame')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/sam3_masks_tracks.png', dpi=150, bbox_inches='tight')
plt.show()


# Per-class instance counts. Only 'person' and 'car' can appear here — those are the two
# concepts we prompted SAM 3 with.
plot_data = []
for frame_idx in range(len(data['n_instances_by_class'])):
    frame_instance_counts = data['n_instances_by_class'][frame_idx]
    frame_class_map = data['classes_by_name'][frame_idx]

    for class_id_key, count in frame_instance_counts.items():
        class_id = int(class_id_key)
        class_name = frame_class_map[class_id] if 0 <= class_id < len(frame_class_map) else f"Unknown_class_{class_id}"
        plot_data.append({'frame_idx': frame_idx, 'class_name': class_name, 'count': count})

df_class_counts = pd.DataFrame(plot_data)

plt.figure(figsize=(15, 7))
sns.histplot(data=df_class_counts, x='frame_idx', hue='class_name', weights='count',
             multiple='stack', binwidth=1, palette=PALETTE_BY_CLASS)
plt.title('Instance Counts per Frame by Class (SAM 3, prompted with person + car)',
          fontsize=16, fontweight='bold')
plt.xlabel('Frame Index', fontsize=12)
plt.ylabel('Total Instances', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print('Mean instances per frame by class:')
print(df_class_counts.groupby('class_name')['count'].agg(['mean', 'min', 'max']).round(1).to_string())

### 3.2.2 Instance Counts per Frame by Class

> ⚠️ Reference-run figures. The per-class means are printed by the cell above for your run.

What the chart shows
---

A stacked bar chart where each bar = one frame, and the two colours are the **two concepts we
prompted for** — nothing else can appear in this chart:

- **Blue** — `car`, consistently detected
- **Red** — `person`, more variable

<u>Cars — highly stable</u>
---

`car` sits between ~15–22 instances across almost all 200 frames with very little variance. This is
a class that is **always present in the scene** — parked and queuing vehicles do not come and go.

<u>Persons — scene-driven variance</u>
---

`person` drives all the variability in the total count. It ranges from ~6 to ~40+ instances and
tracks the scene complexity curve you have seen in the other graphs:

| Frame range | `person` instances | Pattern |
|---|---|---|
| 0–25 | 30–45 | Dense scene, peak early |
| 25–75 | 10–25 | Scene simplifies |
| 75–110 | 5–15 | Minimum complexity |
| 110–135 | 15–20 | Mid recovery |
| 135–200 | 15–25 | Gradual increase |


<u>The spike at the last frames</u>
---

The final frames hit **~86 total instances** — roughly double the typical maximum. This is a clear
outlier. Plausible causes:

- A sudden scene change with many new objects
- Over-segmentation of a cluttered frame (many small, partially visible instances crossing `conf=0.5`)
- An artifact of the video ending (partial frame, encoding artifact triggering false detections)

Notably, **both classes spike** together — `car` jumps from ~20 to ~42, `person` from ~20 to ~44 —
which suggests a genuine scene-level event rather than a class-specific glitch. Worth checking
against the frame itself before believing either story.

<u>Key SAM 3 vs YOLO11 comparison</u>
---

Across the run SAM 3 returned **35–86 instances per frame, mean 37.4** (the figure section 4.2
computes). YOLO11m averaged ~16, yolo11x ~14.

Why the gap? **Not** because SAM 3 "segments everything" — it was prompted with exactly two
concepts at `conf=0.5`, and the chart above contains exactly those two classes. The gap is
**open-vocabulary recall**: SAM 3's concept-conditioned detector finds small, distant, occluded and
partially visible people and cars that YOLO11's COCO-trained head does not clear `conf=0.30` on.

That is a claim about recall, and counting is not evaluating: some of those extra instances are real
recoveries and some are duplicates or false positives. Nothing in this notebook measures which —
see exercise 5.

<a name='summary'></a>
# 4 - YOLO11 (CPU & GPU) vs SAM 3 — Summary

In [ ]:
# Register the SAM 3 run next to the YOLO runs.
# The device key must be 'GPU', matching the YOLO entries — storing it as lowercase 'gpu' made
# matplotlib treat it as a third x category, so SAM 3 got its own empty column.
all_results.pop("sam3", None)
all_results["sam3"] = {"GPU": data}

MODELS_ALL = MODEL_NAMES + ["sam3"]

metrics_keys = ['mean_ms', 'p95_ms', 'fps']
titles   = ['Mean latency (ms)', 'P95 latency (ms)', 'FPS (1000/mean_ms)']
labels   = list(DEVICES.keys())               # x categories: CPU, GPU
x        = np.arange(len(labels))
width    = 0.8 / len(MODELS_ALL)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, key, title in zip(axes, metrics_keys, titles):
    for i, m in enumerate(MODELS_ALL):
        model_base_name = m.split('.')[0].split('-')[0]
        vals = [all_results[m][l]['summary'][key] if l in all_results[m] else np.nan
                for l in labels]
        offset = (i - (len(MODELS_ALL) - 1) / 2) * width
        bars = ax.bar(x + offset, vals, width=width,
                      color=PALETTE_BY_MODEL.get(m, 'steelblue'),
                      edgecolor='white', linewidth=1.2, label=model_base_name)
        ax.bar_label(bars, fmt='%.1f', padding=2, fontsize=8, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_title(title, fontsize=12, fontweight='bold')
    if key != 'fps':
        ax.set_yscale('log')                  # ~18 ms next to ~1500 ms needs a log axis
        ax.set_ylabel('ms (log scale)')
    ax.margins(y=0.25)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

axes[0].legend(frameon=False, fontsize=9)
fig.suptitle('Inference  |  YOLO11 (CPU/GPU) vs SAM 3 (GPU)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/cpu_vs_gpu_vs_sam3.png', dpi=150, bbox_inches='tight')
plt.show()

print('NOTE ON SCOPES: the YOLO `mean_ms` is wall clock including tracking, while SAM 3\'s is the '
      'forward pass only.\nUse the mean_fwd_ms / mean_total_ms columns in the table below for a '
      'like-for-like comparison.')

In [ ]:
# Every number in section 4 comes from this table. Nothing here is typed by hand.
comparison_df = summary_frame(all_results)
print(comparison_df.to_string())

print('\nRanked by mean latency (fastest first):')
print(comparison_df['mean_ms'].sort_values().to_string())


def ratio(num, den, key='mean_fwd_ms'):
    """Latency ratio between two (model, device) rows, on a single stated scope."""
    if num not in comparison_df.index or den not in comparison_df.index:
        return None
    return comparison_df.loc[num, key] / comparison_df.loc[den, key]


print('\nSAM 3 (GPU) vs yolo11x-seg (GPU) — ratios on this protocol:')
for key in ('mean_fwd_ms', 'mean_total_ms', 'mean_ms'):
    r = ratio(('sam3', 'GPU'), ('yolo11x-seg', 'GPU'), key)
    if r is not None:
        print(f'  {key:<15} {r:6.1f}x   '
              f'(SAM 3 {comparison_df.loc[("sam3","GPU"), key]:.1f} ms vs '
              f'{comparison_df.loc[("yolo11x-seg","GPU"), key]:.1f} ms)')

print('\nP95 / mean — tail behaviour, all runs:')
print(comparison_df['p95/mean'].to_string())


## 4.1 The headline number

> ⚠️ **The table printed above is the authority.** The figures repeated in this text are from the
> reference run (Colab T4, 200 frames, `imgsz=640`) and are kept only so the discussion is concrete.
> An earlier version of this notebook hardcoded a GPU P95 of ~20.5 ms against a 45.7 ms mean — a
> value that cannot exist for a right-skewed distribution — and then reasoned from it. Trust the
> generated columns, not remembered numbers.

**SAM 3 on GPU ≈ 1490.7 ms mean forward pass → ~0.7 FPS** in the reference run: **≈33× yolo11x-seg
on GPU, on this protocol.** The cell above recomputes that ratio on three explicit scopes
(`mean_fwd_ms`, `mean_total_ms`, `mean_ms`) so you can see how much the answer depends on which
quantity you pick — that dependence *is* the lesson.

**Running SAM 3 in real time is not feasible on this hardware.**

---

### Mean latency comparison

Read the `mean_ms` / `mean_fwd_ms` columns of the generated table, and the "Ranked by mean latency"
list. The shape of the result in the reference run:

- SAM 3 on **GPU** landed in the same range as the mid-size YOLO models on **CPU**.
- The GPU acceleration that collapses yolo11x from ~1.8 s to ~46 ms does not do the same for SAM 3.

Why not? Not because attention is quadratic over a million pixels — SAM 3's Perception Encoder sees
**5184 patch tokens** (1008/14 = 72 per side) and uses **windowed** attention with global attention
only every 8th layer. The honest reasons are more ordinary and more useful: it is a ~450M-parameter
backbone doing thousands of tokens per frame, plus a DETR decoder over object queries, plus
per-frame memory attention for temporal consistency. That is simply far more arithmetic than one
convolutional pass at 640², and no amount of the *same* GPU changes the ratio between two workloads
that both already use it well.

---

### P95 latency — SAM 3 has tails too

Read the `p95/mean` column. In the reference run SAM 3's P95 sat about 40% above its mean, driven by
frame complexity (the instance-count spike at the end of the clip). Compare it against the GPU YOLO
rows in the same column: the question to answer from your run is not "is SAM 3 slower" — it
obviously is — but *whether it is also less predictable*, because that is what decides whether it
can sit in a pipeline with a fixed frame budget at all.

---

### FPS

From the reference run: yolo11n ~56 FPS on GPU, yolo11m ~43, yolo11x ~22, and SAM 3 ~0.7. Note that
SAM 3 at ~0.7 FPS lands **just above** yolo11x on CPU (~0.5 FPS) — the same order of magnitude —
but it gets there *while occupying a GPU*, which makes it far more expensive per frame in both
hardware and power.

---

### Why GPU doesn't rescue SAM 3

YOLO11 benefits enormously from a GPU because its work is dense, parallel conv/matmul that maps
cleanly onto one. SAM 3's work maps onto a GPU perfectly well too — there is just an order of
magnitude more of it, distributed across a large backbone, a query decoder and a memory bank.

| Model | CPU→GPU speedup (reference run) |
|---|---|
| yolo11n | ~7.5× |
| yolo11m | ~31× |
| yolo11x | ~40× |
| SAM 3 | not measured — GPU only here |

We never ran SAM 3 on CPU, so its CPU→GPU speedup is unknown, not "architecture-limited". If you
want that row filled in, exercise 7 tells you what it costs to find out.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

yolo_11_inference = "results/yolo11_inference_time.png"
sam3_inference = "results/sam3_inference_time.png"

# Open the images
image1 = Image.open(yolo_11_inference)
image2 = Image.open(sam3_inference)

# Display image1 with adapted figsize
plt.figure(figsize=(image1.width / 100, image1.height / 100)) # Convert pixels to inches
plt.imshow(image1)
plt.title('YOLO11 Inference Time')
plt.axis('off') # Turn off axis labels and ticks
plt.show()

# Display image2 with adapted figsize
plt.figure(figsize=(image2.width / 100, image2.height / 100)) # Convert pixels to inches
plt.imshow(image2)
plt.title('SAM3 Inference Time')
plt.axis('off') # Turn off axis labels and ticks
plt.show()

In [ ]:
yolo_11_mask_track = "results/yolo11_masks_tracks.png"
sam3_mask_track = "results/sam3_masks_tracks.png"

# Open the images
image1 = Image.open(yolo_11_mask_track)
image2 = Image.open(sam3_mask_track)

# Display image1 with adapted figsize
plt.figure(figsize=(image1.width / 100, image1.height / 100)) # Convert pixels to inches
plt.imshow(image1)
plt.title('YOLO11 — Detected Masks & Active Tracks')
plt.axis('off')
plt.show()

# Display image2 with adapted figsize
plt.figure(figsize=(image2.width / 100, image2.height / 100)) # Convert pixels to inches
plt.imshow(image2)
plt.title('SAM 3 — Detected Masks & Active Tracks')
plt.axis('off')
plt.show()

In [ ]:
# Detection counts, generated from the runs — the "mean masks/frame" table in 4.2 reads from here.
mask_rows = []
for model, per_device in all_results.items():
    for device, run in per_device.items():
        if 'n_masks' not in run:
            continue
        n = np.asarray(run['n_masks'], dtype=float)
        mask_rows.append({
            'Model': model.split('.')[0],
            'Device': device,
            'Mean masks/frame': round(float(np.mean(n)), 1),
            'Median': round(float(np.median(n)), 1),
            'Min': int(np.min(n)),
            'Max': int(np.max(n)),
        })

masks_df = pd.DataFrame(mask_rows).set_index(['Model', 'Device'])
print(masks_df.to_string())

if ('sam3', 'GPU') in masks_df.index:
    sam = masks_df.loc[('sam3', 'GPU'), 'Mean masks/frame']
    print('\nSAM 3 mean masks/frame relative to each YOLO run:')
    for idx in masks_df.index:
        if idx == ('sam3', 'GPU'):
            continue
        print(f'  vs {idx[0]:<14} {idx[1]:<4} {sam / masks_df.loc[idx, "Mean masks/frame"]:5.1f}x')

## 4.2 Detected Masks & Active Tracks

> ⚠️ The table printed by the cell above is generated from the runs. Figures quoted here are from
> the reference run.

### Mean masks per frame

In the reference run: yolo11n 3.7, yolo11x 13.9, yolo11m 16.0, **SAM 3 37.4** — roughly 2.3× yolo11m
and 10× yolo11n. The generated table recomputes all of these, plus the min/median/max per run, and
prints SAM 3's ratio against each YOLO row.

This is the core behavioural difference, and it is worth stating precisely, because the tempting
explanation is wrong. SAM 3 was **prompted with two concepts** (`person`, `car`) at `conf=0.5`; it
is concept-conditioned, not class-agnostic, and there is no "segment everything" mode in this
benchmark. The extra instances are therefore not background texture, shadows or reflections — they
are **more people and more cars**: small, distant, occluded and partially visible ones that
YOLO11's COCO detection head does not push over `conf=0.30`.

So the right label for the gap is **open-vocabulary recall**, not exhaustiveness. And recall claims
need ground truth: this notebook counts instances, it never checks them, so some fraction of SAM 3's
extra masks are duplicates or false positives. Exercise 5 is where that gets measured.

---

### Scene structure — same video, same wave

The shape of the SAM 3 curve matches what the YOLO11 detection graphs showed, at a different scale:

| Frame range | SAM 3 count | YOLO11m count | Notes |
|---|---|---|---|
| 0–15 | 45–62 | 10–15 | Both peak early |
| 15–35 | Declining to ~30 | Declining | Camera move / scene change |
| 35–75 | 23–44 | 8–15 | Mid-video trough |
| 75–125 | ~25–40 | ~10–18 | Stable, low complexity |
| 125–200 | ~35–45 | ~15–25 | Rising toward the end |

The wave shapes correlate, which is the point: scene complexity is the shared driver across both
models even though the absolute counts differ by ~2.5×.

---

### The early peak (frames 0–15, up to 62)

The spike to **62 masks around frame 10** is the highest point in the sequence, and it lines up with
the ~2100 ms latency plateau over frames 5–25 in the SAM 3 latency plot — more instances to resolve
and to decode masks for means more time per frame. That cross-chart correlation is consistent, and
it is the cleanest evidence in the notebook that per-frame cost is instance-driven, not just
resolution-driven.

---

<a name='conclusions'></a>
# 5 - Conclusions

<u>What this means architecturally</u>
---

SAM 3 **does** condition on a concept: its defining task is Promptable Concept Segmentation, and in
this benchmark it was prompted with `person` and `car`. So the count gap versus YOLO11 is not
"SAM 3 segments everything, including texture and shadows" — every mask it returned belongs to one
of the two concepts we asked for. The gap is **open-vocabulary recall**: SAM 3 finds small,
occluded and partially visible instances of a prompted concept that YOLO11's fixed 80-class head
misses at `conf=0.30`.

**The ~37 mean versus YOLO's ~14 therefore does not make SAM 3 "more accurate" either.** Counting
is not evaluating. What the two numbers do show is that the models answer different questions:

- **YOLO11** — a fixed vocabulary, one convolutional forward pass at 640², masks assembled from 32
  learned prototypes. Fast, and good enough where the classes you care about are in COCO.
- **SAM 3** — an open-vocabulary, promptable, temporally consistent segmenter: a ~450M-parameter
  Perception Encoder backbone (5184 patch tokens, windowed attention, global every 8th layer) feeding
  a DETR-style decoder with a presence head, plus per-frame memory attention. Far higher mask
  fidelity and recall, at ≈33× the per-frame cost **on this protocol**.

They are not competing models — they serve different deployment contexts.

<u>What this means about benchmarking</u>
---

The pedagogical payload of this class is not the ratio; it is the protocol. Three habits to take
away:

1. **State the scope of every latency number.** Wall clock with tracking and a forward-pass-only
   figure differ by a large factor. Compare like with like, or you will publish a speedup that is
   an artefact of your instrumentation.
2. **`torch.cuda.synchronize()` before every timestamp**, and discard warm-up frames. Without both,
   you are timing kernel launches and cuDNN autotuning.
3. **Generate your tables from the measurements.** Every hand-typed figure in the earlier version of
   this notebook eventually contradicted another cell — including a P95 that was smaller than its
   own mean, which is impossible, and which a later cell then used to conclude that GPU latency was
   "highly predictable".

And the point that survives all of it: **more GPU does not fix a workload mismatch.** Choose the
model whose *task* matches yours, then optimise.

<a name='exercises'></a>
# 6 - Think & Exercise

**1. Effect of input resolution on detection count and latency**

Run yolo11m-seg on the same video at `imgsz=320`, `640`, `960`, `1280` on GPU.

* Record mean latency, P95 latency, FPS, and mean detected masks/frame for each
* Plot latency vs imgsz — does it scale quadratically? Fit a curve and check
* At which resolution does detection count stabilize? Is there a point of diminishing returns?

```text
Expected finding: latency scales roughly as imgsz², but mask count plateaus.
The plateau point is where increasing resolution no longer finds new objects.
```

**2. Frame rate subsampling vs. full processing**

* Process a 30 FPS video at every frame, every 2nd frame, and every 5th frame using yolo11x-seg on CPU
* Measure effective throughput (detections per real second of video)
* Compare whether skipped frames cause missed detections on fast-moving objects

```text
This is a classic real-time tradeoff.
A stride of 2–3 often gives near-identical detection coverage for slow scenes at half the compute cost.
```

**3. IoU threshold and NMS sensitivity**

- Find a frame with dense, overlapping objects (from the high-complexity frames in the benchmark)
- Run yolo11m-seg with `iou=0.3`, `0.5`, `0.7`, `0.9` and visualize mask output for each
- **At low IoU, does the model merge adjacent objects?**
- **At high IoU, does it split single objects?**
- Record how detection count changes across IoU values for the same frame

```text
Lower IoU threshold = more aggressive suppression = fewer detections.
Dense scenes need a higher IoU threshold to keep adjacent detections.
This is exactly the regime that caused the latency spikes in the benchmark.
```

**4. How much of the CPU latency tail was NMS? (uses YOLO26)**

The CPU analysis above blames the yolo11x tail on NMS + mask post-processing scaling with detection
count. **YOLO26** (Jan 2026) is natively end-to-end and has no NMS stage at all, which makes it a
direct test rather than an argument.

- Re-run the whole benchmark with `yolo26m-seg.pt` alongside `yolo11m-seg.pt`, same protocol
- Compare `mean_post_ms` and the `p95/mean` ratio on the CPU rows
- **How much of the tail disappears?** If the answer is "most of it", the explanation in 2.3.1 was
  right; if not, the tail is coming from mask upsampling, not NMS

```text
Also worth noting: YOLO26 removes DFL and is reported ~43% faster on CPU ONNX than YOLO11n,
so exercise 1's quadratic-in-imgsz curve is worth re-fitting too.
```

**5. Measure mask quality instead of asserting it (the missing metric)**

This notebook counts masks; it never checks them. "SAM 3 trades speed for mask fidelity" and
"SAM 3 has better recall" are both unmeasured claims.

- Pick 20 frames spanning simple and dense scenes
- For each, compute **mask IoU** between yolo11x-seg and SAM 3 for matched instances (match by box
  IoU first, then compare masks); report mean IoU and the distribution
- For the SAM 3 masks with no YOLO match, inspect 20 of them by hand: how many are real people and
  cars YOLO missed, and how many are duplicates or false positives?
- Optionally label those 20 frames and compute a real mask AP for both models

```text
This is the number that turns section 4.2 from a claim into a result.
Without it, "37.4 vs 13.9" is a count, not a quality comparison.
```

**6. What does SAM 3's native resolution cost?**

We forced `imgsz=640` on SAM 3 so both models saw the same input, but SAM 3's native size is 1008
(patch 14 → 72×72 = 5184 tokens).

- Re-run SAM 3 at `imgsz=1008` and compare `mean_fwd_ms`, mask count, and mask quality against the
  640 run
- Token count scales with (imgsz/14)²: 640 → ~2090 tokens, 1008 → 5184. Does latency scale with
  tokens, with tokens², or somewhere in between? What does that tell you about how much of the
  backbone's attention is windowed rather than global?

**7. Fill in the missing row: SAM 3 on CPU**

Section 4.1 has an empty cell — SAM 3's CPU→GPU speedup was never measured, so nothing in this
notebook actually shows that SAM 3 is "architecture-limited" rather than simply large.

- Run SAM 3 on CPU for **10 frames only** (extrapolate carefully, and say that you did)
- Compare its CPU→GPU ratio against the YOLO models'
- If the ratio is similar to yolo11x's ~40×, then SAM 3 is just a bigger model; if it is much
  smaller, something about its workload really does under-use the GPU. Which is it?

**8. Trackers beyond the two in section 1**

- Re-run the yolo11m-seg GPU benchmark with `bytetrack.yaml`, `botsort.yaml`, `botsort.yaml` with
  `with_reid: True`, and `tracktrack.yaml`
- Record mean latency and the number of distinct track IDs over 200 frames
- Fewer unique IDs for the same objects usually means fewer ID switches. Which tracker pays how much
  latency for that, on a video whose camera barely moves?